# gas-sim-pro · Training Notebook
**Always open fresh from GitHub** — never reuse old Colab tabs.

Run all cells top to bottom. Do not skip cells.
- Cells 1–6: always run
- Cell 6b: optional Optuna search (set `RUN_OPTUNA=True`)
- Cell 7: gate assessment — never stops execution
- Cells 8–11: skip automatically if gate failed


In [ ]:
# ── Cell 1 — Config + auth + registry ───────────────────────────────────
PROJECT_ID = 'gas-sim-pro-ii'
BUCKET     = f'{PROJECT_ID}-gas-sim-data'

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

gcs    = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

def read_registry():
    return json.loads(bucket.blob('model_registry.json').download_as_text())

reg = read_registry()

print('── Registry ──────────────────────────────')
print(f'  last_trained:     {reg["last_trained"]}')
print(f'  last_data_upload: {reg["last_data_upload"]}')
print(f'  feature_version:  {reg["feature_version"]}')
print(f'  current MAE:      {reg["mae"]}')
print(f'  previous MAE:     {reg["previous_mae"]}')
print(f'  rows_trained_on:  {reg["rows_trained_on"]}')
print(f'  gate_status:      {reg["gate_status"]}')
print('──────────────────────────────────────────')


In [ ]:
# ── Cell 2 — Install dependencies ───────────────────────────────────────
!pip install -q xgboost wandb pyarrow pandas-gbq
print('Done.')


In [ ]:
# ── Cell 3 — Load features from GCS ─────────────────────────────────────
import pandas as pd

blobs = list(gcs.bucket(BUCKET).list_blobs(prefix='features/latest/'))
paths = [f'gs://{BUCKET}/{b.name}' for b in blobs if b.name.endswith('.parquet')]
print(f'Found {len(paths)} parquet file(s)')
assert len(paths) > 0, 'No Parquet files found — run cloud.sh option 8 first'

df = pd.concat(
    [pd.read_parquet(p, storage_options={'token': 'google_default'}) for p in paths],
    ignore_index=True
)

assert len(df) >= 500, f'Only {len(df)} rows — need at least 500'
print(f'Loaded {len(df):,} rows · {len(df.columns)} columns')
print(f'Leak count distribution:')
print(df['target_n_leaks'].value_counts().sort_index().to_string())
print()
df[['sensor_delta','centroid_row','centroid_col',
    'n_leaks','target_centroid_row','target_centroid_col',
    'target_nearest_row','target_nearest_col']].describe()


In [ ]:
# ── Cell 4 — Feature matrix + train/val split ────────────────────────────
from sklearn.model_selection import train_test_split
import numpy as np

# Sensor features (19) + multi-leak features (5) = 24 total
FEATURES = [
    # Sensor-derived
    'sensor_delta', 'sensor_mean', 'reading_variance',
    'centroid_row', 'centroid_col', 'coverage_ratio',
    'wind_angle', 'wind_magnitude', 'distance_to_boundary',
    'wind_x', 'wind_y', 'diffusion_rate', 'decay_factor',
    'leak_injection', 'sensor_count',
    'n_sensors_above_threshold', 'max_reading',
    'max_reading_row', 'max_reading_col',
    # Multi-leak
    'n_leaks',
    'leaks_centroid_row', 'leaks_centroid_col',
    'leaks_spread_row', 'leaks_spread_col',
]

# Three prediction targets — train separate models for each
# Primary: centroid (always valid, n_leaks-agnostic)
# Secondary: nearest individual leak to sensor centroid
# Tertiary: leak count
TARGETS_CENTROID = ['target_centroid_row', 'target_centroid_col']
TARGETS_NEAREST  = ['target_nearest_row',  'target_nearest_col']
TARGETS_COUNT    = ['target_n_leaks']

X = df[FEATURES].fillna(0).values.astype(np.float32)

y_centroid = df[TARGETS_CENTROID].values.astype(np.float32)
y_nearest  = df[TARGETS_NEAREST].values.astype(np.float32)
y_count    = df[TARGETS_COUNT].values.astype(np.float32)

X_tr, X_val, yc_tr, yc_val, yn_tr, yn_val, yk_tr, yk_val = train_test_split(
    X, y_centroid, y_nearest, y_count,
    test_size=0.15, random_state=42
)
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')
print(f'Features: {len(FEATURES)}')


In [ ]:
# ── Cell 5 — W&B init (optional, never blocks) ───────────────────────────
import wandb

WANDB_ACTIVE = False
run = None
try:
    run = wandb.init(
        project='gas-sim-pro',
        anonymous='allow',
        mode='offline',
        config={
            'model': 'xgboost',
            'features': FEATURES,
            'n_rows': len(df),
            'feature_version': reg['feature_version'],
        }
    )
    WANDB_ACTIVE = True
    print('W&B run initialised (offline mode)')
except Exception as e:
    print(f'W&B unavailable ({e}) — continuing without tracking')


In [ ]:
# ── Cell 6 — Train three XGBoost models ──────────────────────────────────
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

BASE_PARAMS = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)

print('Training centroid model (primary)...')
model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**BASE_PARAMS))
model_centroid.fit(X_tr, yc_tr)
pred_c = model_centroid.predict(X_val)
mae_centroid = float(np.mean(np.abs(pred_c - yc_val)))
print(f'  Centroid MAE: {mae_centroid:.4f} cells')

print('Training nearest-leak model (secondary)...')
model_nearest = MultiOutputRegressor(xgb.XGBRegressor(**BASE_PARAMS))
model_nearest.fit(X_tr, yn_tr)
pred_n = model_nearest.predict(X_val)
mae_nearest = float(np.mean(np.abs(pred_n - yn_val)))
print(f'  Nearest MAE:  {mae_nearest:.4f} cells')

print('Training count model (tertiary)...')
model_count = xgb.XGBRegressor(**BASE_PARAMS)
model_count.fit(X_tr, yk_tr.ravel())
pred_k = model_count.predict(X_val)
mae_count = float(np.mean(np.abs(pred_k - yk_val.ravel())))
print(f'  Count MAE:    {mae_count:.4f} leaks')

# Primary metric for gate = centroid MAE
mae = mae_centroid

if WANDB_ACTIVE and run:
    wandb.log({'mae_centroid': mae_centroid, 'mae_nearest': mae_nearest, 'mae_count': mae_count})

print(f'\nPrimary metric (centroid MAE): {mae:.4f}')


In [ ]:
# ── Cell 6b — Optuna hyperparameter search (optional) ────────────────────
# Set RUN_OPTUNA=True to search. Runs ~20-40 min on Colab free tier.
# Best params saved to GCS and reused automatically on future runs.

RUN_OPTUNA = False
N_TRIALS   = 40

if RUN_OPTUNA:
    !pip install -q optuna
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 100, 800),
            max_depth        = trial.suggest_int('max_depth', 3, 10),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            subsample        = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
            gamma            = trial.suggest_float('gamma', 0, 5),
            random_state=42, n_jobs=-1, verbosity=0
        )
        m = MultiOutputRegressor(xgb.XGBRegressor(**params))
        m.fit(X_tr, yc_tr)
        pred = m.predict(X_val)
        return float(np.mean(np.abs(pred - yc_val)))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    best = study.best_params
    print(f'Best centroid MAE: {study.best_value:.4f}')
    print(f'Best params: {best}')

    bucket.blob('registry/hparam_best.json').upload_from_string(
        json.dumps({'best_params': best, 'best_mae': study.best_value,
                    'n_trials': N_TRIALS, 'n_rows': len(df),
                    'searched_at': __import__('datetime').datetime.utcnow().isoformat()},
                   indent=2),
        content_type='application/json'
    )
    print('Best params saved to registry/hparam_best.json')

    # Retrain all three models with best params
    model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
    model_centroid.fit(X_tr, yc_tr)
    model_nearest = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
    model_nearest.fit(X_tr, yn_tr)
    model_count = xgb.XGBRegressor(**best, random_state=42, n_jobs=-1)
    model_count.fit(X_tr, yk_tr.ravel())
    pred_c = model_centroid.predict(X_val)
    mae = float(np.mean(np.abs(pred_c - yc_val)))
    print(f'Final centroid MAE with best params: {mae:.4f}')

else:
    try:
        saved = json.loads(bucket.blob('registry/hparam_best.json').download_as_text())
        if saved.get('n_rows', 0) >= len(df) * 0.5:
            best = saved['best_params']
            print(f'Using saved hyperparameters (searched on {saved["n_rows"]:,} rows)')
            model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
            model_centroid.fit(X_tr, yc_tr)
            model_nearest = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
            model_nearest.fit(X_tr, yn_tr)
            model_count = xgb.XGBRegressor(**best, random_state=42, n_jobs=-1)
            model_count.fit(X_tr, yk_tr.ravel())
            pred_c = model_centroid.predict(X_val)
            mae = float(np.mean(np.abs(pred_c - yc_val)))
            print(f'Centroid MAE with saved params: {mae:.4f}')
        else:
            print('Saved params from too few rows — using Cell 6 defaults.')
    except Exception:
        print('No saved hyperparameters — using Cell 6 defaults.')


In [ ]:
# ── Cell 7 — MAE gate (advisory, never stops execution) ──────────────────
reg = read_registry()
prod_mae = reg.get('mae') or float('inf')
prev_mae = reg.get('previous_mae')

print('── MAE Gate Assessment ──────────────────────────────────────')
print(f'  New centroid MAE : {mae:.4f} cells')
print(f'  Current prod     : {prod_mae:.4f} cells')
print(f'  Gate threshold   : {prod_mae * 0.98:.4f}')
print()

if prod_mae == float('inf'):
    GATE_STATUS = 'first_model'
    print('✅ First model — will deploy.')
elif mae < prod_mae * 0.98:
    GATE_STATUS = 'passed'
    print(f'✅ GATE PASSED: {(prod_mae-mae)/prod_mae*100:.1f}% improvement. Will deploy.')
elif mae < prod_mae:
    GATE_STATUS = 'marginal'
    print(f'⚠️  GATE MARGINAL: {(prod_mae-mae)/prod_mae*100:.1f}% better but below 2% threshold.')
    print('   Generate more data and retrain.')
else:
    GATE_STATUS = 'failed'
    print(f'❌ GATE FAILED: {(mae-prod_mae)/prod_mae*100:.1f}% worse than production.')
    print('   Possible causes: stale Parquet, same data, distribution shift.')
    print('   Run cloud.sh option 8 then retry.')
    print('   Cells 8-9 will be skipped. Prod model unchanged.')

if WANDB_ACTIVE and run:
    wandb.log({'gate_status': GATE_STATUS, 'prod_mae': prod_mae})

print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 8 — Export models ───────────────────────────────────────────────
import os, datetime, joblib

if GATE_STATUS not in ('passed', 'first_model'):
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
    VERSION = None
else:
    RUN_ID  = run.id if WANDB_ACTIVE and run else datetime.datetime.now().strftime('%H%M%S')
    VERSION = f"v{datetime.date.today().strftime('%Y%m%d')}-{RUN_ID}"
    os.makedirs(f'/tmp/{VERSION}', exist_ok=True)

    joblib.dump(model_centroid, f'/tmp/{VERSION}/model_centroid.joblib')
    joblib.dump(model_nearest,  f'/tmp/{VERSION}/model_nearest.joblib')
    joblib.dump(model_count,    f'/tmp/{VERSION}/model_count.joblib')
    print(f'✓ Models exported: {VERSION}')
    print(f'  model_centroid.joblib — primary (centroid MAE: {mae:.4f})')
    print(f'  model_nearest.joblib  — nearest leak')
    print(f'  model_count.joblib    — leak count')


In [ ]:
# ── Cell 9 — Push to GCS ─────────────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
else:
    for fname in ['model_centroid.joblib', 'model_nearest.joblib', 'model_count.joblib']:
        blob = bucket.blob(f'models/{VERSION}/{fname}')
        blob.upload_from_filename(f'/tmp/{VERSION}/{fname}')
        print(f'✓ Uploaded: models/{VERSION}/{fname}')

    if WANDB_ACTIVE and run:
        try:
            wandb.log_artifact(f'/tmp/{VERSION}/model_centroid.joblib', name='model-centroid', type='model')
        except Exception as e:
            print(f'  W&B artifact skipped: {e}')


In [ ]:
# ── Cell 10 — Update registry ────────────────────────────────────────────
import datetime as dt

prev_version = reg.get('latest_version')
prev_mae_val = reg.get('mae')

if GATE_STATUS in ('passed', 'first_model') and VERSION is not None:
    reg.update({
        'latest_version':   VERSION,
        'previous_version': prev_version,
        'last_trained':     dt.datetime.now(dt.timezone.utc).isoformat(),
        'model_path':       f'models/{VERSION}/model_centroid.joblib',
        'joblib_path':      f'models/{VERSION}/model_centroid.joblib',
        'mae':              mae,
        'previous_mae':     prev_mae_val,
        'rows_trained_on':  len(df),
        'feature_version':  reg['feature_version'],
        'gate_status':      GATE_STATUS,
        'mae_nearest':      mae_nearest,
        'mae_count':        mae_count,
        'model_nearest_path': f'models/{VERSION}/model_nearest.joblib',
        'model_count_path':   f'models/{VERSION}/model_count.joblib',
        'n_features':       len(FEATURES),
    })
else:
    reg.update({
        'last_trained':    dt.datetime.now(dt.timezone.utc).isoformat(),
        'gate_status':     GATE_STATUS,
        'rows_trained_on': len(df),
    })

reg_blob = bucket.blob('model_registry.json')
reg_blob.upload_from_string(json.dumps(reg, indent=2), content_type='application/json')
reg_blob.cache_control = 'no-cache, no-store, max-age=0'
reg_blob.patch()

result = {
    'status':      GATE_STATUS,
    'mae':         mae,
    'mae_nearest': mae_nearest,
    'mae_count':   mae_count,
    'prod_mae':    prod_mae,
    'version':     VERSION,
    'trained_at':  dt.datetime.now(dt.timezone.utc).isoformat(),
    'rows':        len(df),
    'n_leaks_dist': df['target_n_leaks'].value_counts().to_dict(),
}
r_blob = bucket.blob('registry/last_training_result.json')
r_blob.upload_from_string(json.dumps(result, indent=2, default=str), content_type='application/json')
r_blob.cache_control = 'no-cache, no-store, max-age=0'
r_blob.patch()

if WANDB_ACTIVE and run:
    wandb.finish()

print()
print('── Training Summary ─────────────────────────────────────────')
print(f'  Gate:          {GATE_STATUS}')
print(f'  Centroid MAE:  {mae:.4f}')
print(f'  Nearest MAE:   {mae_nearest:.4f}')
print(f'  Count MAE:     {mae_count:.4f} leaks')
print(f'  Version:       {VERSION}')
print(f'  Rows trained:  {len(df):,}')
print(f'  Leak dist:     {df["target_n_leaks"].value_counts().to_dict()}')
print()
if GATE_STATUS in ('passed', 'first_model'):
    print('🚀 Deploy function will trigger automatically (~5 min).')
    print('   TRAIN button will flash green when deployed.')
elif GATE_STATUS == 'marginal':
    print('⚠️  Below threshold. Generate more data and retrain.')
    print('   TRAIN button will flash yellow.')
else:
    print('❌ Prod model unchanged.')
    print('   Run cloud.sh option 8 then retry.')
    print('   TRAIN button will flash yellow.')
print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 11 — Verify predictions ─────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
else:
    sample = X_val[:5]
    pred_c = model_centroid.predict(sample)
    pred_n = model_nearest.predict(sample)
    pred_k = model_count.predict(sample)
    print('Predictions (first 5 rows):')
    print(f'{"centroid":>20}  {"nearest":>20}  {"count":>6}')
    for i in range(5):
        print(f'  ({pred_c[i][0]:5.1f}, {pred_c[i][1]:5.1f})'
              f'  ({pred_n[i][0]:5.1f}, {pred_n[i][1]:5.1f})'
              f'  {pred_k[i]:.1f} leaks')
    print('Verification complete.')
